In [1]:
import csv
from pathlib import Path
DATA_DIR = Path.cwd().parent / 'data'
raw_path = DATA_DIR / 'orders_raw.csv'

with open(raw_path, encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))

row = rows[2]
print(row)

problem = {'order_id':row['order_id'], 
           'field':'amount', 
           'raw_value':row['amount'], 
           'reason':'не число'} 
print(problem)
import pprint
pprint.pprint(rows)

{'order_id': '103', 'city': '', 'amount': 'abc'}
{'order_id': '103', 'field': 'amount', 'raw_value': 'abc', 'reason': 'не число'}
[{'amount': '1 200', 'city': ' казань ', 'order_id': '101'},
 {'amount': '800', 'city': 'Москва', 'order_id': '102'},
 {'amount': 'abc', 'city': '', 'order_id': '103'},
 {'amount': '2 500', 'city': 'санкт-петербург', 'order_id': '104'}]


In [2]:
print(Path.cwd() / 'data' / 'orders_raw.csv')
print(Path.cwd().parent / 'data' / 'orders_raw.csv')
print((Path.cwd() / 'data' / 'orders_raw.csv').exists())
print((Path.cwd().parent / 'data' / 'orders_raw.csv').exists())

c:\Projects\ml-journey\stage-1-python\data\orders_raw.csv
c:\Projects\ml-journey\data\orders_raw.csv
False
True


In [3]:
import json
from pathlib import Path

DATA_DIR = Path.cwd().parent / 'data'
json_path = DATA_DIR / 'example.json'

report = [
    {'order_id': '103', 'field': 'amount', 'raw_value': 'abc', 'reason': 'не число'},
]

# запись: сначала объект, потом файл
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# чтение
with open(json_path, encoding='utf-8') as f:
    loaded = json.load(f)

print(type(loaded))          # <class 'list'>
print(loaded[0]['reason'])   # не число

<class 'list'>
не число


In [4]:
import json
print(json.dumps(problem))
print(json.dumps(problem, ensure_ascii=False, indent=2))

{"order_id": "103", "field": "amount", "raw_value": "abc", "reason": "\u043d\u0435 \u0447\u0438\u0441\u043b\u043e"}
{
  "order_id": "103",
  "field": "amount",
  "raw_value": "abc",
  "reason": "не число"
}


In [7]:
from orders_utils import parse_amount
problems = []

for row in rows:       
    if parse_amount(row['amount']) is None:
        temp_dict = {'order_id':row['order_id'], 
           'field':'amount', 
           'raw_value':row['amount'], 
           'reason':'не число'}
        problems.append(temp_dict)
    if row['city'] == '':
        temp_dict = {'order_id':row['order_id'], 
                   'field':'city', 
                   'raw_value':row['city'], 
                   'reason':'пусто'}
        problems.append(temp_dict)        
  
    

problems_path = Path.cwd().parent / 'data' / 'problems.json'

with open(problems_path, 'w', encoding='utf-8') as f:
    json.dump(problems, f, ensure_ascii=False, indent=2)




In [10]:
with open(problems_path, encoding='utf-8') as f:
    loaded = json.load(f)
print(type(loaded))
print(len(loaded))
print(loaded)

<class 'list'>
2
[{'order_id': '103', 'field': 'amount', 'raw_value': 'abc', 'reason': 'не число'}, {'order_id': '103', 'field': 'city', 'raw_value': '', 'reason': 'пусто'}]


In [12]:
s = json.dumps({'city': '', 'amount': None, 'text': 'null'})
print(s)
back = json.loads(s)
print(back)
print(back['amount'] is None, back['text'] is None)

{"city": "", "amount": null, "text": "null"}
{'city': '', 'amount': None, 'text': 'null'}
True False


In [ ]:
import csv
from pathlib import Path

DATA_DIR = Path.cwd().parent / 'data'
REQUIRED = {'order_id', 'city', 'amount'}   # обязательные столбцы — множество


def read_orders(path: Path) -> list[dict]:
    """Читает CSV с заказами. Падает с ValueError, если нет обязательных столбцов."""
    with open(path, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        missing = REQUIRED - set(reader.fieldnames)   # fieldnames — список названий столбцов
        if missing:
            raise ValueError(f'в файле {path.name} нет столбцов: {missing}')
        return list(reader)


try:
    rows = read_orders(DATA_DIR / 'orders_raw.csv')
except FileNotFoundError as e:
    print(f'файл не найден: {e.filename}')
except ValueError as e:
    print(f'плохой файл: {e}')
else:
    print(f'прочитано строк: {len(rows)}')
finally:
    print('проверка файла завершена')

прочитано строк: 4
проверка файла завершена


In [6]:
bad_path = DATA_DIR / 'orders_bad.csv'
with open(bad_path, 'w', encoding='utf-8') as f:
    f.write('id,city,amount\n101,Казань,1200\n')

try:
    rows = read_orders(bad_path)
except FileNotFoundError as e:
    print(f'файл не найден: {e.filename}')
except ValueError as e:
    print(f'плохой файл: {e}')
else:
    print(f'прочитано строк: {len(rows)}')
finally:
    print('проверка файла завершена')

плохой файл: в файле orders_bad.csv нет столбцов: {'order_id'}
проверка файла завершена


In [8]:
row = {'id': '101', 'city': 'Казань', 'amount': '1200'}
print(row['order_id'])

KeyError: 'order_id'

In [7]:
def safe_amount(value):
    try:
        result = float(value)
    except ValueError:
        print('ошибка')
        return None
    else:
        print('ок')
        return result
    finally:
        print('готово')

print(safe_amount('12.5'))
print(safe_amount('abc'))

ок
готово
12.5
ошибка
готово
None
